# BRD4 BD1/BD2 — MD preparation workflow and command archive

**CHPC Lengau | CHEM1937 | 27 November 2026**

This is the separate MD preparation notebook, preserving the commands and scripts used from the docking-to-MD handover through CGenFF parameter generation and quality assessment. The earlier docking and ligand-audit phase is documented in [Complete docking analysis](../COMPLETE_DOCKING_ANALYSIS/BRD4_Complete_Docking_Analysis_Executed.ipynb); the CGenFF figures, parsed source ZIPs and tables are in [the executed parameter audit](CGENFF/BRD4_CGenFF_Parameter_Audit_Executed.ipynb).

**How to use:** copy the Bash cells into the CHPC terminal in sequence, following the notes. Windows CMD commands and web-interface instructions are Markdown. The Bash kernel is optional; do not use Run All. Module activation and environment variables must persist in the same terminal. Commands are archived from the conversation, not rerun in this notebook; successful terminal output is recorded separately as observations. The failed direct RDKit MOL2 validation is retained as a historical diagnostic, not an approved step. Paths refer to the actual CHPC workspace.

### Scope and decisions

- Natural products: NP01 Psychotrine (+1), NP02 Isosinensetin (0), NP09 Sciadopitysin (0).
- Reference: WSH (0), the common crystal ligand in 2YEL and 2YEM.
- Eight systems: each of the four ligands in BRD4-BD1 and BRD4-BD2.
- Original validated docking grids only for MD starting poses; NP01 uses the corrected Meeko protonated poses; NP02 and NP09 use original docking poses that passed basic typing QC; WSH uses the crystal pose in its respective domain.
- Planned force-field family: CHARMM36m/CGenFF 5.0, with GROMACS as the MD engine. The prior Plasmepsin II–Sesamin workflow is a reference, not a substitute for BRD4-specific validation.
- This notebook stops at parameter generation/audit. Topology conversion, system assembly, minimisation, equilibration and production MD have not yet been executed or approved.
- No docking PDBQT partial charges will be reused for MD.


## 1. Backup and software discovery (docking-to-MD handover)

The archive below was created on CHPC before MD preparation. The command only checks the local archive; an offsite transfer and checksum comparison must be verified independently. It does not include the separate screening data on Lustre.

In [ ]:
cd ~

ARCHIVE="BRD4_NP_DOCKING_$(date +%Y%m%d_%H%M%S).tar.gz"

tar -czf "$ARCHIVE" BRD4_NP_DOCKING && \
gzip -t "$ARCHIVE" && \
sha256sum "$ARCHIVE" > "$ARCHIVE.sha256" && \
ls -lh "$ARCHIVE" "$ARCHIVE.sha256"

Recorded archive: `BRD4_NP_DOCKING_20261127_060438.tar.gz` (1.4 MB) and its SHA-256 sidecar. An offsite checksum match was not supplied in the conversation.

Windows CMD (archived transfer commands):

```cmd
mkdir "%USERPROFILE%\BRD4_BACKUP"
cd /d "%USERPROFILE%\BRD4_BACKUP"
scp apeter1@scp.chpc.ac.za:~/BRD4_NP_DOCKING_20261127_060438.tar.gz .
scp apeter1@scp.chpc.ac.za:~/BRD4_NP_DOCKING_20261127_060438.tar.gz.sha256 .
certutil -hashfile BRD4_NP_DOCKING_20261127_060438.tar.gz SHA256
type BRD4_NP_DOCKING_20261127_060438.tar.gz.sha256
```

In [ ]:
module avail 2>&1 | grep -Ei \
'gromacs|amber|antechamber|acpype|openmm|packmol'

Observed modules included AMBER20, GROMACS 2024 CPU, GROMACS 2024.1 CUDA and OpenMM 7.5. Availability of a module is not proof of executable access.

## 2. Historical AMBER20 access diagnosis (not the selected MD path)

These commands were used to diagnose the original AMBER plan. AMBER access was blocked by group permissions, motivating reuse of the previously successful CHARMM/CGenFF workflow.

In [ ]:
deactivate

module purge

module load chpc/amber/20/mpich-3.4.3/gcc-9.2.0

echo "AMBERHOME=$AMBERHOME"

for tool in antechamber parmchk2 tleap sqm cpptraj
do
    printf "%-15s " "$tool"
    command -v "$tool" || echo "NOT FOUND"
done

In [ ]:
ls -lh "$AMBERHOME/dat/leap/cmd/leaprc.gaff2"
ls -lh "$AMBERHOME/dat/leap/cmd/leaprc.protein.ff14SB"
ls -lh "$AMBERHOME/dat/leap/cmd/leaprc.water.tip3p"

module list
module show chpc/amber/20/mpich-3.4.3/gcc-9.2.0 2>&1
echo "AMBERHOME=$AMBERHOME"

ls -ld "$AMBERHOME"
ls -ld "$AMBERHOME/bin"
ls -lah "$AMBERHOME/bin" | head -40
echo "$PATH" | tr ':' '\n' | grep -i amber

for tool in antechamber parmchk2 tleap sqm cpptraj
do
    ls -l "$AMBERHOME/bin/$tool"
done

id
groups
namei -l "$AMBERHOME/bin/antechamber"

**Observed:** `AMBERHOME=/home/apps/chpc/chem/amber/20`; the PATH contained its `bin` directory, but all five tools returned `NOT FOUND` or `Permission denied`. The bin directory had mode `drwxrwx---` and group `amber`; the account belonged to `apeter1` and `gpu`, not `amber`. Actual GPU execution was not yet verified.

Alternative installations were proposed for inspection (results not supplied):

In [ ]:
for V in 14 18 20
do
    echo "===== AMBER $V ====="
    ls -ld "/home/apps/chpc/chem/amber/$V/bin"
    test -x "/home/apps/chpc/chem/amber/$V/bin/antechamber" \
        && echo "ACCESSIBLE" || echo "NOT ACCESSIBLE"
done

## 3. Selected CHARMM/CGenFF workflow — environment

The archived previous workflow used CHARMM36m/CGenFF 5.0 and GROMACS. Relevant prior notebooks: [GALAXY_3.ipynb](https://colab.research.google.com/drive/1tFPrJ72YyfnkxS_QsRKPmG402nZL4492) and [1LF2_Sesamin_Colab_MD.ipynb](https://drive.google.com/file/d/1ulMqqr_aaxxxisrKKy89auGZ19obxlkm/view). These are private Drive references; no credentials are included here. Confirm the exact compatible force-field and converter versions before topology conversion.

In [ ]:
cd ~/BRD4_NP_DOCKING

module purge
module load python/3.9.6
module load chpc/openbabel/3.1.1/cmake-3.20.0/gcc-6.1.0

source ~/brd4_meeko_env/bin/activate

mkdir -p 10_MD/{CGENFF_INPUT,CGENFF_RETURN,FORCEFIELD,LIGAND_PARAMS,SYSTEMS,QC,SCRIPTS}

python -c "import rdkit; print(rdkit.__version__)"

obabel -V

## 4. Ligand input generation — exact preparation script

This successful script preserves source connectivity, charge and heavy-atom coordinates, reconstructs the docked coordinates via element-constrained assignment and adds explicit H atoms. WSH uses its prepared BD1 crystal SDF for parameter generation. This is not the final coordinate assembly for all eight complexes.

In [ ]:
cat > 10_MD/SCRIPTS/prepare_cgenff.py <<'PY'
from pathlib import Path
from rdkit import Chem
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import numpy as np
import subprocess

out = Path("10_MD/CGENFF_INPUT")

elements = {
    "A":"C", "C":"C", "N":"N", "NA":"N",
    "O":"O", "OA":"O", "S":"S", "SA":"S",
    "F":"F", "Cl":"Cl", "Br":"Br", "I":"I"
}

def read_atoms(path):
    atoms = []
    for l in Path(path).read_text().splitlines():
        if l.startswith("ENDMDL"):
            break
        if not l.startswith(("ATOM  ","HETATM")):
            continue
        t = l.split()[-1]
        if t in ("H","HD","HS"):
            continue
        atoms.append((
            int(l[6:11]), t,
            [float(l[i:i+8]) for i in (30,38,46)]
        ))
    return atoms

for name, domain, expected in [
    ("NP01","BD1",1),
    ("NP02","BD2",0),
    ("NP09","BD1",0),
    ("WSH","BD1",0)
]:

    if name == "WSH":
        sdf = Path(
            "08_PROTONATION/PH75/REDOCKING/"
            "BD1_WSH_prepared.sdf"
        )
        prep = pose = None

    else:
        sdf = Path(f"03_LIGANDS/PH75/SDF/{name}.sdf")

        if name == "NP01":
            b = Path("05_DOCKING/NP01_MEEKO_TEST")
            prep = b/"PDBQT/NP01_protonated.pdbqt"
            pose = b/"POSES/BD1_protonated.pdbqt"
        else:
            prep = Path(
                f"03_LIGANDS/PH75/PDBQT/{name}.pdbqt"
            )
            pose = Path(
                f"05_DOCKING/PH75/POSES/{domain}/{name}.pdbqt"
            )

    mol = Chem.SDMolSupplier(
        str(sdf), removeHs=False
    )[0]

    assert mol is not None
    mol = Chem.RemoveHs(mol)

    assert Chem.GetFormalCharge(mol) == expected

    identity = Chem.MolToInchiKey(mol)

    if pose is not None:
        a,b = read_atoms(prep),read_atoms(pose)
        n = mol.GetNumAtoms()

        assert len(a) == len(b) == n

        assert [(x[0],x[1]) for x in a] == [
            (x[0],x[1]) for x in b
        ]

        d = cdist(
            mol.GetConformer().GetPositions(),
            np.array([x[2] for x in a])
        )

        for i,atom in enumerate(mol.GetAtoms()):
            for j,item in enumerate(a):
                if atom.GetSymbol() != elements[item[1]]:
                    d[i,j] = 1e6

        r,c = linear_sum_assignment(d)
        assert np.max(d[r,c]) <= 0.02

        conf = Chem.Conformer(n)

        for i,j in zip(r,c):
            conf.SetAtomPosition(int(i),b[j][2])

        mol.RemoveAllConformers()
        mol.AddConformer(conf)

    heavy_xyz = mol.GetConformer().GetPositions().copy()

    mol = Chem.AddHs(mol,addCoords=True)

    assert Chem.GetFormalCharge(mol) == expected
    assert Chem.MolToInchiKey(Chem.RemoveHs(mol)) == identity

    assert np.allclose(
        heavy_xyz,
        mol.GetConformer().GetPositions()[:len(heavy_xyz)],
        atol=1e-6
    )

    assert np.isfinite(
        mol.GetConformer().GetPositions()
    ).all()

    mol.SetProp("_Name",name)

    sdfout = out/f"{name}.sdf"
    mol2out = out/f"{name}.mol2"

    writer = Chem.SDWriter(str(sdfout))
    writer.write(mol)
    writer.close()

    subprocess.run([
        "obabel",str(sdfout),
        "-O",str(mol2out)
    ],check=True)

    assert mol2out.stat().st_size > 0

    print(f"\n===== {name} =====")
    print("Source:",sdf)
    print("InChIKey:",identity)
    print("Formal charge:",expected)
    print("Heavy atoms:",mol.GetNumHeavyAtoms())
    print("Total atoms:",mol.GetNumAtoms())
    print("Heavy-atom coordinates: PRESERVED")
    print("SDF identity: PASS")
    print("MOL2 generated:",mol2out)

print("\nCGENFF INPUT GENERATION: COMPLETE")
PY

python 10_MD/SCRIPTS/prepare_cgenff.py \
    2>&1 | tee 10_MD/QC/cgenff_preparation.log

**Recorded successful CHPC results**

| Ligand | InChIKey | Charge | Heavy atoms | All atoms |
|---|---|---:|---:|---:|
| NP01 | CLNILJPYOXIKRO-REIDKSKDSA-O | +1 | 34 | 71 |
| NP02 | UYCWETIUOAGWIL-UHFFFAOYSA-N | 0 | 27 | 47 |
| NP09 | YCXRBCHEOFVYEN-UHFFFAOYSA-N | 0 | 43 | 67 |
| WSH | TUWDLUFFAHHNEF-QHCPKHFHSA-N | 0 | 32 | 53 |

All SDF identities passed; heavy-atom coordinates preserved. The generated MOL2 files required separate validation.

## 5. Historical direct RDKit MOL2 validation attempt — failed (do not use as acceptance test)

Archived for reproducibility. All four Open Babel MOL2 files failed direct RDKit sanitisation. NP01 showed a nitrogen-valence error; the other three showed kekulisation errors. This is consistent with parser/typing convention differences, not proof that the molecules themselves are incorrect. Skip this block when repeating the accepted workflow; it is retained as the original diagnostic.

```bash
cd ~/BRD4_NP_DOCKING

python - <<'PY'
from pathlib import Path
from rdkit import Chem
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import numpy as np

base = Path("10_MD/CGENFF_INPUT")
out = Path("10_MD/CGENFF_SUBMIT")
out.mkdir(exist_ok=True)

passed = 0

for name, expected in [
    ("NP01", 1),
    ("NP02", 0),
    ("NP09", 0),
    ("WSH", 0)
]:
    print(f"\n===== {name} =====")

    try:
        sdf = Chem.SDMolSupplier(
            str(base/f"{name}.sdf"),
            removeHs=False
        )[0]

        mol2 = Chem.MolFromMol2File(
            str(base/f"{name}.mol2"),
            sanitize=True,
            removeHs=False
        )

        assert sdf is not None, "SDF parsing failed"
        assert mol2 is not None, "MOL2 parsing failed"

        key1 = Chem.MolToInchiKey(Chem.RemoveHs(sdf))
        key2 = Chem.MolToInchiKey(Chem.RemoveHs(mol2))

        charge = Chem.GetFormalCharge(mol2)

        print("SDF InChIKey:", key1)
        print("MOL2 InChIKey:", key2)
        print("MOL2 formal charge:", charge)
        print("SDF atoms:", sdf.GetNumAtoms())
        print("MOL2 atoms:", mol2.GetNumAtoms())

        assert key1 == key2, "Identity mismatch"
        assert charge == expected, "Charge mismatch"
        assert sdf.GetNumAtoms() == mol2.GetNumAtoms()
        assert sdf.GetNumBonds() == mol2.GetNumBonds()

        a = sdf.GetConformer().GetPositions()
        b = mol2.GetConformer().GetPositions()

        assert np.isfinite(b).all()

        d = cdist(a,b)

        for i,x in enumerate(sdf.GetAtoms()):
            for j,y in enumerate(mol2.GetAtoms()):
                if x.GetAtomicNum() != y.GetAtomicNum():
                    d[i,j] = 1e6

        r,c = linear_sum_assignment(d)

        assert d[r,c].max() <= 0.02

        mapping = dict(zip(r,c))

        for bond in sdf.GetBonds():
            i = mapping[bond.GetBeginAtomIdx()]
            j = mapping[bond.GetEndAtomIdx()]

            mb = mol2.GetBondBetweenAtoms(int(i),int(j))

            assert mb is not None, "Connectivity mismatch"
            assert bond.GetBondType() == mb.GetBondType(), \
                "Bond-order mismatch"

        # Assign unique atom names for topology mapping.
        lines = (base/f"{name}.mol2").read_text().splitlines()
        section = ""
        count = 0
        names = []

        for i,line in enumerate(lines):
            if line.startswith("@<TRIPOS>"):
                section = line.strip()
                continue

            if section == "@<TRIPOS>ATOM" and line.strip():
                fields = line.split()
                count += 1

                symbol = mol2.GetAtomWithIdx(
                    count-1
                ).GetSymbol()

                atom_name = f"{symbol}{count}"
                fields[1] = atom_name
                names.append(atom_name)

                lines[i] = " ".join(fields)

        assert count == mol2.GetNumAtoms()
        assert len(set(names)) == count

        target = out/f"{name}.mol2"
        target.write_text("\n".join(lines)+"\n")

        print("Identity: PASS")
        print("Formal charge: PASS")
        print("Connectivity: PASS")
        print("Bond orders: PASS")
        print("Coordinates: PASS")
        print("Unique atom names: PASS")
        print("SUBMISSION FILE:", target)

        passed += 1

    except Exception as e:
        print("QC REVIEW REQUIRED:", e)

print(f"\nCGENFF INPUT QC: {passed}/4 PASSED")

if passed != 4:
    raise SystemExit(1)
PY
```

Observed: `CGENFF INPUT QC: 0/4 PASSED`.

## 6. Accepted MOL2 round-trip validation — exact successful commands

Open Babel reads its own MOL2 representation and writes a round-trip SDF. RDKit compares this SDF with the original SDF. The original MOL2 is retained for submission, with unique atom names; the round-trip file is a QC artifact.

In [ ]:
cd ~/BRD4_NP_DOCKING

mkdir -p 10_MD/CGENFF_ROUNDTRIP

for LIG in NP01 NP02 NP09 WSH
do
    obabel \
        "10_MD/CGENFF_INPUT/${LIG}.mol2" \
        -O "10_MD/CGENFF_ROUNDTRIP/${LIG}.sdf"
done

In [ ]:
python - <<'PY'
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
import numpy as np

base = Path("10_MD/CGENFF_INPUT")
rt = Path("10_MD/CGENFF_ROUNDTRIP")
out = Path("10_MD/CGENFF_SUBMIT")
out.mkdir(exist_ok=True)

passed = 0

for name, expected in [
    ("NP01",1), ("NP02",0),
    ("NP09",0), ("WSH",0)
]:
    print(f"\n===== {name} =====")

    try:
        a = Chem.SDMolSupplier(
            str(base/f"{name}.sdf"),
            removeHs=False
        )[0]

        b = Chem.SDMolSupplier(
            str(rt/f"{name}.sdf"),
            removeHs=False
        )[0]

        assert a is not None and b is not None

        ka = Chem.MolToInchiKey(Chem.RemoveHs(a))
        kb = Chem.MolToInchiKey(Chem.RemoveHs(b))

        qa = Chem.GetFormalCharge(a)
        qb = Chem.GetFormalCharge(b)

        print("Original identity:",ka)
        print("Roundtrip identity:",kb)
        print("Original charge:",qa)
        print("Roundtrip charge:",qb)
        print("Original formula:",
              rdMolDescriptors.CalcMolFormula(a))
        print("Roundtrip formula:",
              rdMolDescriptors.CalcMolFormula(b))

        assert qa == qb == expected, "Charge mismatch"
        assert ka == kb, "Identity mismatch"
        assert a.GetNumAtoms() == b.GetNumAtoms()
        assert a.GetNumBonds() == b.GetNumBonds()

        xyz1 = a.GetConformer().GetPositions()
        xyz2 = b.GetConformer().GetPositions()

        assert np.isfinite(xyz2).all()

        d = cdist(xyz1,xyz2)

        for i,x in enumerate(a.GetAtoms()):
            for j,y in enumerate(b.GetAtoms()):
                if x.GetAtomicNum() != y.GetAtomicNum():
                    d[i,j] = 1e6

        r,c = linear_sum_assignment(d)

        maximum = d[r,c].max()
        assert maximum <= 0.02

        mapping = dict(zip(r,c))

        for bond in a.GetBonds():
            i = int(mapping[bond.GetBeginAtomIdx()])
            j = int(mapping[bond.GetEndAtomIdx()])

            other = b.GetBondBetweenAtoms(i,j)

            assert other is not None
            assert bond.GetBondType() == other.GetBondType()

        print("Maximum coordinate difference:",
              round(maximum,6),"A")

        lines = (base/f"{name}.mol2").read_text().splitlines()
        section = ""
        count = 0
        names = set()

        for i,line in enumerate(lines):

            if line.startswith("@<TRIPOS>"):
                section = line.strip()
                continue

            if section == "@<TRIPOS>ATOM" and line.strip():
                fields = line.split()
                count += 1

                element = fields[5].split(".")[0]
                atom_name = f"{element}{count}"

                assert atom_name not in names
                assert len(atom_name) <= 4

                names.add(atom_name)
                fields[1] = atom_name
                lines[i] = " ".join(fields)

                if element == "N":
                    print("Nitrogen MOL2 record:",
                          " ".join(fields))

        assert count == a.GetNumAtoms()

        target = out/f"{name}.mol2"
        target.write_text("\n".join(lines)+"\n")

        print("Chemical identity: PASS")
        print("Formal charge: PASS")
        print("Bond orders: PASS")
        print("Coordinates: PASS")
        print("Unique atom names: PASS")
        print("Output:",target)

        passed += 1

    except Exception as e:
        print("QC REVIEW REQUIRED:",e)

print(f"\nROUNDTRIP QC: {passed}/4 PASSED")

if passed != 4:
    raise SystemExit(1)
PY

**Observed successful results: ROUNDTRIP QC: 4/4 PASSED**

| Ligand | Formula | Formal charge | Maximum coordinate deviation (Å) |
|---|---|---:|---:|
| NP01 | C28H37N2O4+ | +1 | 0.000 |
| NP02 | C20H20O7 | 0 | 0.000 |
| NP09 | C33H24O10 | 0 | 0.000 |
| WSH | C25H21N5O2 | 0 | 0.000 |

All four retained InChIKey, connectivity, atom count and bond-order identity in the round-trip SDF. NP01 protonated tertiary nitrogen was N5 (MOL2 type N.4); its other N34 was N.pl3. These checks validate the input representation but do not establish force-field accuracy.

## 7. Windows file transfer and organisation

The following commands were used to transfer the four submission files to Windows and move the folder to Downloads. These are Windows CMD commands, not Bash.

```cmd
mkdir "%USERPROFILE%\BRD4_CGENFF"
scp apeter1@scp.chpc.ac.za:~/BRD4_NP_DOCKING/10_MD/CGENFF_SUBMIT/*.mol2 "%USERPROFILE%\BRD4_CGENFF"
dir "%USERPROFILE%\BRD4_CGENFF"
move "%USERPROFILE%\BRD4_CGENFF" "%USERPROFILE%\Downloads\"
explorer "%USERPROFILE%\Downloads\BRD4_CGENFF"
```

Recorded file sizes: NP01 5,331 bytes; NP02 3,567 bytes; NP09 5,090 bytes; WSH 4,034 bytes. Four files transferred, 18,022 bytes in total. The move was instructed; its terminal output was not supplied.

## 8. CGenFF web parameter generation

Web application: [CGenFF](https://cgenff.com/). Submit each validated MOL2 individually and download the ZIP result. No API credentials or account data are needed in this notebook.

| Option | Intended selection |
|---|---|
| Force-field version | 5.0 |
| Output Mol2 | Enabled |
| Include Warnings | Enabled |
| Include Debug Info | Enabled |
| Copy Existing Params | Disabled |

The WSH initial settings are archived in [WSH_submission_settings.md](CGENFF/WSH_submission_settings.md). The subsequent WSH and NP09 job-info screenshots listed Copy Existing Params, although their initial intended settings were off; the exact final toggle state must be confirmed from job metadata rather than assumed. No claim is made that every option was identical across all jobs.

Sequence: Start CGenFF Job → upload MOL2 → choose version/options → Process Input → Run CGenFF Engine → Download CGenFF Results.

The four result ZIPs (NP01, NP02, NP09, WSH) were supplied by the user and parsed in the separate self-contained executed audit notebook. See the audit for raw source archives, file hashes, topology/charge tables and figures.

## 9. Parameter quality assessment

| Ligand | Atom count | Net charge | Maximum parameter penalty | Maximum charge penalty |
|---|---:|---:|---:|---:|
| NP01 | 71 | +1 | 29.000 | 13.754 |
| NP02 | 47 | 0 | 38.000 | 8.947 |
| NP09 | 67 | 0 | 38.000 | 8.967 |
| WSH | 53 | 0 | 59.000 | 21.132 |

WSH requires targeted validation of its high-penalty ring dihedral. Other flagged parameters also require appropriate validation. Penalties are indicators of analogy uncertainty, not measured parameter errors. Basic structural QC and successful CGenFF output are not sufficient for production MD.

**Executed analysis scripts and figures:** [BRD4_CGenFF_Parameter_Audit_Executed.ipynb](CGENFF/BRD4_CGenFF_Parameter_Audit_Executed.ipynb).

## 10. Next work — not yet performed

1. Obtain the compatible CHARMM36m/CGenFF 5.0 force-field package and topology converter; verify versions and licensing.
2. Transfer returned parameter files to CHPC; verify identity, charges and atom mapping.
3. Validate flagged CGenFF parameters, especially WSH ring torsion, using suitable reference calculations before approval.
4. Convert topology and assemble eight domain-specific complexes with crystal/reference or validated docking poses.
5. Solvation, ionisation, energy minimisation, NVT/NPT, GPU benchmark, and production MD only after the required checks.

No new execution results are represented in this command archive. No earlier repository files were modified.